In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1: Config
# ═══════════════════════════════════════════════════════════════════════

# ========== IMPORTS ==========
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import torchvision.transforms as T
import torchvision.models as models

import pandas as pd
import numpy as np
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score

import warnings
warnings.filterwarnings('ignore')

# ========== PATHS ==========
DATA_ROOT = Path("/kaggle/input/datasets/ashery/chexpert")
TRAIN_CSV = DATA_ROOT / "train.csv"
VALID_CSV = DATA_ROOT / "valid.csv"
IMAGE_ROOT = DATA_ROOT
OUTPUT_DIR = Path("/kaggle/working")
CHECKPOINT_PATH = OUTPUT_DIR / "best_efficientnet_b3.pth"

# ========== OPTIMIZED CONFIG ==========
MODEL_NAME = "efficientnet_b3"
NUM_CLASSES = 14
IMAGE_SIZE = 300  # OPTIMIZED: EfficientNet-B3 native resolution (300x300)
PRETRAINED = True

# Training settings
BATCH_SIZE = 32  # OPTIMIZED: EfficientNet-B3 needs more memory than DenseNet
EPOCHS = 15  # Same as DenseNet for fair comparison
LEARNING_RATE = 5e-4  # Same as DenseNet
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4

# Better validation split
USE_CUSTOM_SPLIT = True
VAL_SPLIT_RATIO = 0.05

# Mixed precision for speed
USE_AMP = True  # Automatic Mixed Precision (30% faster)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ========== LABELS ==========
LABEL_NAMES = [
    "No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly",
    "Lung Opacity", "Lung Lesion", "Edema", "Consolidation",
    "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion",
    "Pleural Other", "Fracture", "Support Devices"
]

UNCERTAIN_POLICY = "ones"  # Treat uncertain as positive

print("="*70)
print("OPTIMIZED EFFICIENTNET-B3 TRAINING")
print("="*70)
print(f"Device: {DEVICE}")
print(f"Image size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Mixed precision: {USE_AMP}")
print("="*70)

OPTIMIZED EFFICIENTNET-B3 TRAINING
Device: cuda
Image size: 300x300
Batch size: 32
Epochs: 15
Learning rate: 0.0005
Mixed precision: True


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2: Dataset with Strong Augmentation
# ═══════════════════════════════════════════════════════════════════════

class CheXpertDataset(Dataset):
    def __init__(self, csv_file, image_root, transform=None, uncertain_policy="ones"):
        self.df = pd.read_csv(csv_file) if isinstance(csv_file, (str, Path)) else csv_file
        self.image_root = Path(image_root)
        self.transform = transform
        self._process_labels(uncertain_policy)
        print(f"✓ Loaded {len(self.df)} samples")

    def _process_labels(self, policy):
        self.df[LABEL_NAMES] = self.df[LABEL_NAMES].fillna(0)
        if policy == "zeros":
            self.df[LABEL_NAMES] = self.df[LABEL_NAMES].replace(-1, 0)
        elif policy == "ones":
            self.df[LABEL_NAMES] = self.df[LABEL_NAMES].replace(-1, 1)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        raw_path = row["Path"]
        if raw_path.startswith("CheXpert-v1.0-small/"):
            relative_path = raw_path.replace("CheXpert-v1.0-small/", "")
        else:
            relative_path = raw_path
        
        img_path = self.image_root / relative_path
        
        if not img_path.exists():
            raise FileNotFoundError(f"Image not found: {img_path}")
        
        image = Image.open(img_path).convert("RGB")
        labels = torch.tensor(row[LABEL_NAMES].values.astype(np.float32))
        
        if self.transform:
            image = self.transform(image)
        
        return image, labels


def get_transforms(train=True):
    """Augmentation optimized for 300px (EfficientNet-B3)"""
    if train:
        return T.Compose([
            T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(10),
            T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
            T.ColorJitter(brightness=0.1, contrast=0.1),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    else:
        return T.Compose([
            T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])


# Create train/val split
print("\nCreating train/val split...")
if USE_CUSTOM_SPLIT:
    full_train_df = pd.read_csv(TRAIN_CSV)
    train_df, val_df = train_test_split(
        full_train_df, 
        test_size=VAL_SPLIT_RATIO, 
        random_state=42,
        shuffle=True
    )
    print(f"✓ Train: {len(train_df)}, Val: {len(val_df)}")
else:
    train_df = pd.read_csv(TRAIN_CSV)
    val_df = pd.read_csv(VALID_CSV)

train_ds = CheXpertDataset(train_df, IMAGE_ROOT, get_transforms(True), UNCERTAIN_POLICY)
val_ds = CheXpertDataset(val_df, IMAGE_ROOT, get_transforms(False), UNCERTAIN_POLICY)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"✓ Train batches: {len(train_loader)}")
print(f"✓ Val batches: {len(val_loader)}")


Creating train/val split...
✓ Train: 212243, Val: 11171
✓ Loaded 212243 samples
✓ Loaded 11171 samples
✓ Train batches: 6633
✓ Val batches: 350


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3: EfficientNet-B3 Model with Class Weighting
# ═══════════════════════════════════════════════════════════════════════

class CheXpertEfficientNet(nn.Module):
    """EfficientNet-B3 optimized for chest X-ray"""
    def __init__(self, num_classes=14, pretrained=True):
        super().__init__()
        backbone = models.efficientnet_b3(
            weights=models.EfficientNet_B3_Weights.DEFAULT if pretrained else None
        )
        in_features = backbone.classifier[1].in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),  # Slightly higher dropout for bigger model
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        feats = self.backbone(x)
        logits = self.classifier(feats)
        return logits


# Compute class weights
def compute_pos_weights(loader):
    pos_counts = torch.zeros(NUM_CLASSES)
    total = 0
    
    print("\nComputing class weights...")
    for _, labels in tqdm(loader, desc="Scanning"):
        pos_counts += labels.sum(dim=0)
        total += labels.size(0)
    
    pos_freq = pos_counts / total
    pos_weights = (total - pos_counts) / (pos_counts + 1e-5)
    
    print("\nClass distribution:")
    for i, name in enumerate(LABEL_NAMES):
        print(f"  {name:30s}: {pos_freq[i]:.3f} (weight: {pos_weights[i]:.2f})")
    
    return pos_weights


model = CheXpertEfficientNet(NUM_CLASSES, PRETRAINED).to(DEVICE)
pos_weights = compute_pos_weights(train_loader).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# BETTER SCHEDULER: Cosine annealing with restarts
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=1, eta_min=1e-6)

# Mixed precision scaler
scaler = torch.amp.GradScaler('cuda') if USE_AMP else None

print(f"\n✓ Model: EfficientNet-B3 (~12M parameters)")
print(f"✓ Loss: Weighted BCE (class-balanced)")
print(f"✓ Optimizer: Adam (LR={LEARNING_RATE})")
print(f"✓ Scheduler: CosineAnnealingWarmRestarts")
if USE_AMP:
    print(f"✓ Mixed Precision: Enabled (30% faster)")

Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 151MB/s]



Computing class weights...


Scanning: 100%|██████████| 6633/6633 [26:46<00:00,  4.13it/s]


Class distribution:
  No Finding                    : 0.100 (weight: 8.98)
  Enlarged Cardiomediastinum    : 0.104 (weight: 8.63)
  Cardiomegaly                  : 0.157 (weight: 5.37)
  Lung Opacity                  : 0.497 (weight: 1.01)
  Lung Lesion                   : 0.048 (weight: 19.91)
  Edema                         : 0.292 (weight: 2.42)
  Consolidation                 : 0.191 (weight: 4.25)
  Pneumonia                     : 0.111 (weight: 7.99)
  Atelectasis                   : 0.300 (weight: 2.33)
  Pneumothorax                  : 0.101 (weight: 8.89)
  Pleural Effusion              : 0.438 (weight: 1.28)
  Pleural Other                 : 0.028 (weight: 35.31)
  Fracture                      : 0.043 (weight: 22.08)
  Support Devices               : 0.524 (weight: 0.91)

✓ Model: EfficientNet-B3 (~12M parameters)
✓ Loss: Weighted BCE (class-balanced)
✓ Optimizer: Adam (LR=0.0005)
✓ Scheduler: CosineAnnealingWarmRestarts
✓ Mixed Precision: Enabled (30% faster)


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4: Training Function
# ═══════════════════════════════════════════════════════════════════════

def train_epoch(model, loader, criterion, optimizer, scaler=None):
    """Train one epoch with optional mixed precision"""
    model.train()
    total_loss = 0
    pbar = tqdm(loader, desc="Training")
    
    for images, labels in pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        if scaler:  # Mixed precision
            with torch.amp.autocast('cuda'):
                logits = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:  # Standard training
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(loader)


def validate(model, loader, criterion):
    """Validate and compute metrics"""
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validation"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            if USE_AMP:
                with torch.amp.autocast('cuda'):
                    logits = model(images)
                    loss = criterion(logits, labels)
            else:
                logits = model(images)
                loss = criterion(logits, labels)
            
            total_loss += loss.item()
            probs = torch.sigmoid(logits)
            all_preds.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    
    avg_loss = total_loss / len(loader)
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    
    return avg_loss, all_preds, all_labels


def compute_auroc(y_true, y_pred):
    """Compute mean AUROC across all labels"""
    aurocs = []
    for i in range(NUM_CLASSES):
        if y_true[:, i].sum() > 0:
            try:
                auroc = roc_auc_score(y_true[:, i], y_pred[:, i])
                aurocs.append(auroc)
            except:
                pass
    return np.mean(aurocs) if aurocs else 0.0


In [5]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5: Training Loop with Early Stopping
# ═══════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print(f"STARTING TRAINING: {EPOCHS} epochs")
print("="*70 + "\n")

best_auroc = 0.0
best_val_loss = float('inf')
patience_counter = 0
PATIENCE = 5  # Stop if no improvement for 5 epochs

history = {'train_loss': [], 'val_loss': [], 'auroc': []}

for epoch in range(1, EPOCHS + 1):
    print(f"\n{'='*70}")
    print(f"Epoch {epoch}/{EPOCHS}")
    print('='*70)
    
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, scaler)
    
    # Validate
    val_loss, val_preds, val_labels = validate(model, val_loader, criterion)
    mean_auroc = compute_auroc(val_labels, val_preds)
    
    # Step scheduler
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    # Log
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['auroc'].append(mean_auroc)
    
    print(f"\nResults:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss: {val_loss:.4f}")
    print(f"  AUROC: {mean_auroc:.4f}")
    print(f"  LR: {current_lr:.6f}")
    
    # Save best model
    if mean_auroc > best_auroc:
        best_auroc = mean_auroc
        best_val_loss = val_loss
        patience_counter = 0
        
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_loss': val_loss,
            'auroc': mean_auroc,
            'history': history
        }, CHECKPOINT_PATH)
        
        print(f"  ✓ NEW BEST MODEL SAVED! (AUROC: {mean_auroc:.4f})")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{PATIENCE})")
    
    # Early stopping
    if patience_counter >= PATIENCE:
        print(f"\n⚠ Early stopping triggered at epoch {epoch}")
        break

print("\n" + "="*70)
print("TRAINING COMPLETE!")
print(f"Best AUROC: {best_auroc:.4f}")
print(f"Best Val Loss: {best_val_loss:.4f}")
print("="*70)



STARTING TRAINING: 15 epochs


Epoch 1/15


Validation: 100%|██████████| 350/350 [01:37<00:00,  3.58it/s]



Results:
  Train Loss: 0.9467
  Val Loss: 0.9311
  AUROC: 0.7501
  LR: 0.000452
  ✓ NEW BEST MODEL SAVED! (AUROC: 0.7501)

Epoch 2/15


Validation: 100%|██████████| 350/350 [00:47<00:00,  7.43it/s]



Results:
  Train Loss: 0.9198
  Val Loss: 0.8985
  AUROC: 0.7693
  LR: 0.000328
  ✓ NEW BEST MODEL SAVED! (AUROC: 0.7693)

Epoch 3/15


...


Results:
  Train Loss: 0.8866
  Val Loss: 0.8767
  AUROC: 0.7809
  LR: 0.000173
  No improvement (3/5)

Epoch 14/15


Validation: 100%|██████████| 350/350 [00:37<00:00,  9.34it/s]



Results:
  Train Loss: 0.8742
  Val Loss: 0.8645
  AUROC: 0.7872
  LR: 0.000049
  No improvement (4/5)

Epoch 15/15


Validation: 100%|██████████| 350/350 [00:40<00:00,  8.74it/s]



Results:
  Train Loss: 0.8626
  Val Loss: 0.8526
  AUROC: 0.7927
  LR: 0.000500
  ✓ NEW BEST MODEL SAVED! (AUROC: 0.7927)

TRAINING COMPLETE!
Best AUROC: 0.7927
Best Val Loss: 0.8526


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6: Final Evaluation
# ═══════════════════════════════════════════════════════════════════════

def compute_detailed_metrics(y_true, y_pred):
    aurocs, aps = {}, {}
    
    for i, name in enumerate(LABEL_NAMES):
        if y_true[:, i].sum() == 0:
            continue
        try:
            aurocs[name] = roc_auc_score(y_true[:, i], y_pred[:, i])
            aps[name] = average_precision_score(y_true[:, i], y_pred[:, i])
        except:
            pass
    
    return aurocs, aps


# Load best model
print("\nLoading best checkpoint...")
checkpoint = torch.load(CHECKPOINT_PATH)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✓ Loaded best model from epoch {checkpoint['epoch']}")
print(f"  AUROC: {checkpoint['auroc']:.4f}")

# Final evaluation
print("\nRunning final evaluation...")
val_loss, val_preds, val_labels = validate(model, val_loader, criterion)
aurocs, aps = compute_detailed_metrics(val_labels, val_preds)

print("\n" + "="*70)
print("FINAL RESULTS - PER-LABEL PERFORMANCE")
print("="*70)
for name in LABEL_NAMES:
    if name in aurocs:
        print(f"{name:30s}: AUROC={aurocs[name]:.4f} AP={aps[name]:.4f}")

print("\n" + "="*70)
print(f"MEAN AUROC: {np.mean(list(aurocs.values())):.4f}")
print(f"MEAN AP: {np.mean(list(aps.values())):.4f}")
print("="*70)

print(f"\n✓ Model saved at: {CHECKPOINT_PATH}")
print("✓ Download from Output tab →")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 7: Model Info (Download Instructions)
# ═══════════════════════════════════════════════════════════════════════

print(f"Model saved at: {CHECKPOINT_PATH}")
print("Download 'best_efficientnet_b3.pth'")
print(f"\nModel size: {CHECKPOINT_PATH.stat().st_size / (1024**2):.2f} MB")
print(f"Final AUROC: {best_auroc:.4f}")
